# Messy Data Issues Report  
**Dataset:** `lahore_flats_v2.csv`

---

## 1. Completeness Issues
Significant missing values were found across multiple critical fields:

- **Price / Marla / Area (sqft):** 599 missing each  
- **Floor Number:** 1,503 missing  
- **Total Floors:** 1,468 missing  
- **Built Year:** 1,117 missing  
- **Description:** 599 missing  
- **Features:** 652 missing  
- **Nearby Locations and Other Facilities:** 789 missing  
- **Other Rooms:** 866 missing  
- **Property ID:** Not found for 599 records  
- **Name:** Appears as `#NAME?` in 155 records  

---

## 2. Validity Issues
Presence of outliers and clearly invalid values:

- **Floor Number:** Includes extreme values such as `1234` and `2018`  
- **Total Floors:** Invalid entries like `2018` and `2020`  
- **Built Year:** Contains `1`, `20242`, and boolean `TRUE` (110 rows affected)  
- **Parking Spaces:** Unrealistic values up to `500`  

---

## 3. Accuracy Issues
Type and format inconsistencies affecting data reliability:

- **Price:** Stored as text with currency symbols/units in ~2,500 rows  
- **Floor Number:** Contains boolean values (`TRUE/FALSE`) in 500 rows  
- **Total Floors:** Non-numeric entries in 278 rows  
- **Built Year:** Non-YYYY formats in 78 rows  

---

## 4. Consistency Issues
Conflicting representations and structural inconsistencies:

- **Duplicate `Link` values:** 20 duplicates identified  
- **Boolean vs numeric mixing:**  
  - Columns such as *Parking Spaces*, *Floor*, *Floors in Building*, *Elevators* mix `TRUE/FALSE` with numeric counts  
- **Area mismatch:**  
  - Area (sqft) never aligns with standard Marla-to-sqft conversion  
- **Column naming issue:**  
  - `\ufeffProperty ID` includes a BOM (Byte Order Mark)  

---

## 5. Messy Data Structure Issues

### 5.1 Multi-Valued Cells
- **Features:** Multiple items packed using `|` delimiter  
- **Nearby Locations and Other Facilities:** Comma-separated lists in one cell  
- **Other Rooms:** Comma-separated values stored in a single column  

### 5.2 Structured Data Embedded as Text
- **Rooms:** Stored as a stringified dictionary, packing multiple attributes into one cell  

### 5.3 Sparse Section Columns
- **Section - Plot Features:** Completely empty in 3,098 rows  
- **Other `Section - ...` columns:** High levels of missingness across the dataset  

---

## 6. Summary
The dataset exhibits major issues across **completeness, validity, accuracy, consistency, and structure**.  
Substantial cleaning, normalization, type enforcement, and schema redesign are required before it can be reliably used for analysis or modeling.

---


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [130]:
df=pd.read_csv("E:\Project\Data Cleaning\lahore_flats_v2.csv")

In [132]:
df.sample(10)

,Property ID,Name,Price,Marla,Area (sqft),Floor Number,Total Floors,Built Year,Address,Description,...,Park Facing,Disputed,File,Balloted,Sewerage,Electricity,Water Supply,Sui Gas,Boundary Wall,Section - Plot Features
482,53534314,"4 BHK Askari 11, Askari, Lahore, Punjab",PKR 3.7 Crore,12.0,2700.0,1,1,NaN,"Askari 11, Askari, Lahore, Punjab",This Flat is based at a prime location of Aska...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1871,Not found,Not listed,NaN,NaN,NaN,NaN,NaN,NaN,Not listed,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1739,Not found,Not listed,NaN,NaN,NaN,NaN,NaN,NaN,Not listed,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1280,52851379,"2 BHK Gulberg, Lahore, Punjab",PKR 2.6 Crore,4.6,1035.0,14,14,2016,"Gulberg, Lahore, Punjab","Located on Main Boulevard Gulberg, the most pr...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2008,53292120,"1 BHK Lake City, Raiwind Road, Lahore, Punjab",PKR 64.64 Lakh,1.8,405.0,2,9,2015,"Lake City, Raiwind Road, Lahore, Punjab",Damax Marketing Offers Luxury Apartment Availa...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2913,53229364,#NAME?,PKR 1.52 Crore,1.8,405.0,TRUE,TRUE,2029,"Sukh Chayn Gardens, Lahore, Punjab",Zameen Jade Project Overview Location: Situate...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1991,Not found,Not listed,NaN,NaN,NaN,NaN,NaN,NaN,Not listed,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2751,53546921,"2 BHK Defence View Apartments, Shanghai Road, ...",PKR 1.55 Crore,5.7,1282.5,NaN,NaN,2025,"Defence View Apartments, Shanghai Road, Lahore...","Khurram Developers Offers, LUXURY 2 BEDROOM AP...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
166,52796407,"3 BHK Askari 10 - Sector F, Askari 10, Askari,...",PKR 3.65 Crore,10.0,2250.0,1,3,2021,"Askari 10 - Sector F, Askari 10, Askari, Lahor...",Brand New 3 Bed ( 10 Marla *2280SQFT)* For SAL...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
723,53421512,"3 BHK Askari 11 - Sector B Apartments, Askari ...",PKR 2.8 Crore,10.0,2250.0,TRUE,9,TRUE,"Askari 11 - Sector B Apartments, Askari 11, As...",Lahore Havalian Estate Offers: 10 Marla luxury...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Analyze issues with data through dynamic & Programetic assessment.

In [133]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3099 entries, 0 to 3098
Data columns (total 84 columns):
 #   Column                                           Non-Null Count  Dtype  
---  ------                                           --------------  -----  
 0   Property ID                                      3099 non-null   object 
 1   Name                                             3099 non-null   object 
 2   Price                                            2500 non-null   object 
 3   Marla                                            2500 non-null   float64
 4   Area (sqft)                                      2500 non-null   float64
 5   Floor Number                                     1596 non-null   object 
 6   Total Floors                                     1631 non-null   object 
 7   Built Year                                       1982 non-null   object 
 8   Address                                          3099 non-null   object 
 9   Description                   

In [134]:
df.describe()

,Marla,Area (sqft)
count,2500.000000,2500.000000
mean,6.248960,1406.016000
std,3.988644,897.444834
min,0.400000,90.000000
25%,2.400000,540.000000
50%,5.000000,1125.000000
75%,10.000000,2250.000000
max,28.500000,6412.500000


## Issues in data

In [5]:
import re
import numpy as np

def price_to_pkr(x):
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return np.nan
    
    s = str(x).strip().lower()
    s = s.replace(",", "")
    
    # common cleanup
    s = re.sub(r"\s+", " ", s)          # normalize spaces
    s = s.replace("pkr", "").strip()    # remove currency label
    
    # extract number + unit
    m = re.search(r"(\d+(?:\.\d+)?)\s*(lakh|crore)", s)
    if not m:
        return np.nan
    
    val = float(m.group(1))
    unit = m.group(2)
    
    if unit == "lakh":
        return val * 100_000
    if unit == "crore":
        return val * 10_000_000
    
    return np.nan

# create numeric column
df["Price"] = df["Price"].apply(price_to_pkr)

# (optional) make it integer rupees where possible
df["Price"] = df["Price"].round().astype("Int64")


In [7]:
df['Price']

0        5400000
1        2157000
2        8000000
3        2000000
4        2500000
          ...   
3094     7500000
3095    12200000
3096     6380000
3097     4500000
3098     5500000
Name: Price, Length: 3099, dtype: Int64

In [8]:
df.columns

Index(['Property ID', 'Name', 'Price', 'Marla', 'Area (sqft)', 'Floor Number',
       'Total Floors', 'Built Year', 'Address', 'Description', 'Features',
       'Nearby Locations and Other Facilities', 'Rooms', 'Other Rooms', 'Link',
       'Parking Spaces', 'Lobby in Building', 'Double Glazed Windows',
       'Central Air Conditioning', 'Central Heating', 'Flooring',
       'Electricity Backup', 'Waste Disposal', 'Floor', 'Floors in Building',
       'Elevators', 'Service Elevators in Building', 'Other Main Features',
       'Broadband Internet Access', 'Satellite or Cable TV Ready',
       'Business Center or Media Room in Building',
       'Conference Room in Building', 'Intercom', 'ATM Machines',
       'Other Business and Communication Facilities',
       'Community Lawn or Garden', 'Community Swimming Pool', 'Community Gym',
       'First Aid or Medical Centre', 'Day Care Centre', 'Kids Play Area',
       'Barbeque Area', 'Mosque', 'Community Centre',
       'Other Community Faci

In [9]:
df["Area (sqft)"] = df["Marla"] * 272.25

In [11]:
df["price_per_sqft"] = df["Price"] / df["Area (sqft)"]

In [12]:
df.columns

Index(['Property ID', 'Name', 'Price', 'Marla', 'Area (sqft)', 'Floor Number',
       'Total Floors', 'Built Year', 'Address', 'Description', 'Features',
       'Nearby Locations and Other Facilities', 'Rooms', 'Other Rooms', 'Link',
       'Parking Spaces', 'Lobby in Building', 'Double Glazed Windows',
       'Central Air Conditioning', 'Central Heating', 'Flooring',
       'Electricity Backup', 'Waste Disposal', 'Floor', 'Floors in Building',
       'Elevators', 'Service Elevators in Building', 'Other Main Features',
       'Broadband Internet Access', 'Satellite or Cable TV Ready',
       'Business Center or Media Room in Building',
       'Conference Room in Building', 'Intercom', 'ATM Machines',
       'Other Business and Communication Facilities',
       'Community Lawn or Garden', 'Community Swimming Pool', 'Community Gym',
       'First Aid or Medical Centre', 'Day Care Centre', 'Kids Play Area',
       'Barbeque Area', 'Mosque', 'Community Centre',
       'Other Community Faci

In [15]:
df = df[df["Property ID"].str.strip().str.lower() != "not found"]



In [16]:
dups = df[df.duplicated(subset=["Link"], keep=False)] \
         .sort_values("Link")

dups[["Link", "Price_PKR", "Area (sqft)", "Bedrooms"]]


KeyError: "['Price_PKR', 'Bedrooms'] not in index"

In [ ]:
df = df.drop_duplicates(subset=["Link"], keep="first")

'https://www.zameen.com/Property/shanghai_road_defence_view_apartments_tower-facing_elegance_2-bed_luxury_apartment_for_sale_in_defence_view_high-end_amenities_best_value-52187608-12643-1.html'

In [74]:
dups = df[df.duplicated(subset=["Link"], keep=False)] \
         .sort_values("Link")

dups[["Link", "Price_PKR", "Area (sqft)", "Bedrooms"]]


,Link,Price_PKR,Area (sqft),Bedrooms


In [17]:
df.shape

(2500, 85)

In [18]:
df.duplicated().sum()

np.int64(0)

In [101]:
df.columns

Index(['Property ID', 'Name', 'Price', 'Marla', 'Area (sqft)', 'Floor Number',
       'Total Floors', 'Built Year', 'Address', 'Description', 'Features',
       'Nearby Locations and Other Facilities', 'Rooms', 'Other Rooms', 'Link',
       'Parking Spaces', 'Lobby in Building', 'Double Glazed Windows',
       'Central Air Conditioning', 'Central Heating', 'Flooring',
       'Electricity Backup', 'Waste Disposal', 'Floor', 'Floors in Building',
       'Elevators', 'Service Elevators in Building', 'Other Main Features',
       'Broadband Internet Access', 'Satellite or Cable TV Ready',
       'Business Center or Media Room in Building',
       'Conference Room in Building', 'Intercom', 'ATM Machines',
       'Other Business and Communication Facilities',
       'Community Lawn or Garden', 'Community Swimming Pool', 'Community Gym',
       'First Aid or Medical Centre', 'Day Care Centre', 'Kids Play Area',
       'Barbeque Area', 'Mosque', 'Community Centre',
       'Other Community Faci

In [96]:
import ast

df['Rooms_dict'] = df['Rooms'].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else {}
)


In [ ]:
rooms_df = pd.json_normalize(df['Rooms_dict'])
rooms_df

,Bedrooms,Bathrooms,Kitchens,Servant Quarters,Store Rooms
0,1,1,1,NaN,NaN
1,1,1,1,NaN,NaN
2,1,1,1,NaN,NaN
3,NaN,1,1,1,1
4,1,1,1,1,1
...,...,...,...,...,...
2495,NaN,NaN,NaN,NaN,NaN
2496,1,2,1,NaN,NaN
2497,1,1,NaN,2,1
2498,1,1,1,1,1


In [100]:
df = pd.concat([df, rooms_df], axis=1)

In [83]:
features = df['Features'].apply(lambda x: x.split("|") if isinstance(x, str) else x)

features


0       [Parking Spaces ,  Lobby in Building ,  Double...
1       [Central Air Conditioning ,  Central Heating ,...
2       [Parking Spaces ,  Lobby in Building ,  Double...
3       [Built in year : 2023 ,  Parking Spaces : 1 , ...
4       [Built in year : 2023 ,  Parking Spaces ,  Lob...
                              ...                        
3094    [Built in year : 2020 ,  Parking Spaces : 20 ,...
3095    [Built in year : 2028 ,  Parking Spaces : 3 , ...
3096    [Built in year : 2025 ,  Parking Spaces ,  Lob...
3097    [Built in year : 2022 ,  Parking Spaces : 4 , ...
3098    [Built in year : 2001 ,  Parking Spaces : 40 ,...
Name: Features, Length: 2500, dtype: object

In [104]:
df['Rooms'][90]

"{'Bedrooms': '3', 'Bathrooms': '4', 'Servant Quarters': '1', 'Kitchens': '1', 'Store Rooms': '1'}"

In [107]:
df=df.drop(columns='Rooms')

In [112]:
df['Nearby Locations and Other Facilities'].isna().sum()

np.int64(681)

In [116]:
df.columns

Index(['Property ID', 'Name', 'Price', 'Marla', 'Area (sqft)', 'Floor Number',
       'Total Floors', 'Built Year', 'Address', 'Description', 'Features',
       'Nearby Locations and Other Facilities', 'Other Rooms', 'Link',
       'Parking Spaces', 'Lobby in Building', 'Double Glazed Windows',
       'Central Air Conditioning', 'Central Heating', 'Flooring',
       'Electricity Backup', 'Waste Disposal', 'Floor', 'Floors in Building',
       'Elevators', 'Service Elevators in Building', 'Other Main Features',
       'Broadband Internet Access', 'Satellite or Cable TV Ready',
       'Business Center or Media Room in Building',
       'Conference Room in Building', 'Intercom', 'ATM Machines',
       'Other Business and Communication Facilities',
       'Community Lawn or Garden', 'Community Swimming Pool', 'Community Gym',
       'First Aid or Medical Centre', 'Day Care Centre', 'Kids Play Area',
       'Barbeque Area', 'Mosque', 'Community Centre',
       'Other Community Facilities', 

'Built Year', 'Address', 'Description', 'Features',
       'Nearby Locations and Other Facilities', 'Other Rooms', 'Link',
       'Parking Spaces', 'Lobby in Building', 'Double Glazed Windows',
       'Central Air Conditioning', 'Central Heating', 'Flooring',
       'Electricity Backup', 'Waste Disposal', 'Floor', 'Floors in Building',
       'Elevators', 'Service Elevators in Building', 'Other Main Features',
       'Broadband Internet Access', 'Satellite or Cable TV Ready',
       'Business Center or Media Room in Building',
       'Conference Room in Building', 'Intercom', 'ATM Machines',
       'Other Business and Communication Facilities',
       'Community Lawn or Garden', 'Community Swimming Pool', 'Community Gym',
       'First Aid or Medical Centre', 'Day Care Centre', 'Kids Play Area',
       'Barbeque Area', 'Mosque', 'Community Centre',
       'Other Community Facilities', 'Sauna', 'Jacuzzi',
       'Other Healthcare and Recreation Facilities', 'Nearby Schools',
       'Nearby Hospitals', 'Nearby Shopping Malls', 'Nearby Restaurants',
       'Distance From Airport (kms)', 'Nearby Public Transport Service',
       'Other Nearby Places', 'Maintenance Staff', 'Security Staff',
       'Laundry or Dry Cleaning Facility', 'Facilities for Disabled',
       'Pets Allowed', 'Other Facilities', 'Section - Main Features',
       'Section - Rooms', 'Section - Business and Communication',
       'Section - Community Features', 'Section - Healthcare Recreational',
       'Section - Nearby Locations and Other Facilities',
       'Section - Other Facilities', 'Furnished', 'Communal/Shared Kitchen',
       'Built in year', 'Pets Not Allowed', 'Possesion', 'Corner',
       'Park Facing', 'Disputed', 'File', 'Balloted', 'Sewerage',
       'Electricity', 'Water Supply', 'Sui Gas', 'Boundary Wall',
       'Section - Plot Features', 'price_per_sqft', 'fe', 'Rooms_dict',
       'Bedrooms', 'Bathrooms', 'Kitchens', 'Servant Quarters', 'Store Rooms'

In [ ]:
df.drop(columns='Property ID')

(2991, 91)

In [127]:
df.loc[df['Built Year'] == 'TRUE', 'Built Year'] = np.nan